# 03 Train YOLOv8n on Google Colab

This notebook trains the deep-learning detector for PorkSort on a free Colab GPU and saves
everything the rest of the project needs (weights, ONNX exports, metrics and per-image
predictions) to Google Drive.

**Before you run it**

1. `Runtime > Change runtime type > T4 GPU`.
2. Upload `Pork Rasher Error-Packaging-.v4i.yolov8.zip` to Google Drive at `MyDrive/MMA3001/`.
3. Either set `REPO_URL` below to the GitHub repo (if it is private, add a Colab secret called
   `GITHUB_TOKEN` holding a read-only token), or leave it and upload `porksort.zip` to `MyDrive/MMA3001/`.
4. `Runtime > Run all`. Training takes roughly 30 to 45 minutes on a T4.

Run it once with `SPLIT = "blocked"` (the main result). If there is time, run it again with
`SPLIT = "roboflow"` so the report can show how much the leaky Roboflow split inflates the score.

When it finishes, download `MyDrive/MMA3001/porksort_runs/yolov8n_<split>.zip` and unzip it into
`results/yolo/` in the repo.

In [ ]:
# ---- Settings ----
SPLIT = "blocked"          # "blocked" (main) or "roboflow" (leakage comparison)
EPOCHS = 60                # early stopping (PATIENCE) usually ends it sooner
PATIENCE = 20
IMGSZ = 640                # network input size; 720x540 frames are letterboxed to 640x480
BATCH = 16
SEED = 0
REPO_URL = "https://github.com/<owner>/<repo>.git"   # <- set this
DRIVE_DIR = "/content/drive/MyDrive/MMA3001"
ZIP_NAME = "Pork Rasher Error-Packaging-.v4i.yolov8.zip"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib
zip_path = pathlib.Path(DRIVE_DIR) / ZIP_NAME
assert zip_path.exists(), f"Upload the dataset zip to {zip_path}"
os.makedirs("/content/data/pork_v4", exist_ok=True)
!unzip -q -o "{zip_path}" -d /content/data/pork_v4
!ls /content/data/pork_v4

In [ ]:
# Get the PorkSort code (same split CSV and helpers as the local runs).
# Either clone from GitHub (set REPO_URL) or upload porksort.zip to DRIVE_DIR.
import subprocess
subprocess.run(["rm", "-rf", "/content/porksort"])
if "<owner>" not in REPO_URL:
    url = REPO_URL
    try:
        from google.colab import userdata
        tok = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@")
    except Exception:
        pass  # public repo, no token needed
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/porksort"], check=True)
else:
    code_zip = pathlib.Path(DRIVE_DIR) / "porksort.zip"
    assert code_zip.exists(), f"Set REPO_URL or upload porksort.zip to {code_zip}"
    subprocess.run(["unzip", "-q", "-o", str(code_zip), "-d", "/content/porksort"], check=True)
assert pathlib.Path("/content/porksort/splits/blocked_split.csv").exists()
!pip -q install "ultralytics>=8.3,<9"
!pip -q install --no-deps -e /content/porksort

In [ ]:
import json, platform, time
import torch, ultralytics
from porksort.config import CLASS_NAMES
from porksort.dataset import build_index
from porksort.splits import load_split, export_yolo_dataset

env = {"ultralytics": ultralytics.__version__, "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "python": platform.python_version(), "split": SPLIT, "imgsz": IMGSZ, "epochs": EPOCHS,
       "batch": BATCH, "seed": SEED}
print(env)

index = build_index("/content/data/pork_v4")
df = load_split(index, SPLIT, csv_path="/content/porksort/splits/blocked_split.csv")
print(df.groupby("split")["fail"].agg(["size", "sum"]))
yaml_path = export_yolo_dataset(df, f"/content/yolo_{SPLIT}", CLASS_NAMES)
print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO
run_name = f"yolov8n_{SPLIT}"
model = YOLO("yolov8n.pt")            # COCO-pretrained weights, fine-tuned below
t0 = time.time()
model.train(data=str(yaml_path), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
            seed=SEED, deterministic=True, project="/content/runs", name=run_name,
            exist_ok=True, plots=True, verbose=True)
env["train_seconds"] = time.time() - t0
run_dir = pathlib.Path("/content/runs") / run_name
best = run_dir / "weights" / "best.pt"
print("best weights:", best, "train time (min):", env["train_seconds"] / 60)

In [ ]:
# Detection metrics on val and test (mAP is a detection metric; pass/fail is evaluated locally)
m = YOLO(str(best))
metrics = {}
for sp in ("val", "test"):
    r = m.val(data=str(yaml_path), split=sp, imgsz=IMGSZ, batch=BATCH, conf=0.001, iou=0.6,
              plots=True, project=str(run_dir), name=f"eval_{sp}", exist_ok=True)
    metrics[sp] = {
        "mAP50": float(r.box.map50), "mAP50_95": float(r.box.map),
        "precision": float(r.box.mp), "recall": float(r.box.mr),
        "per_class_mAP50_95": {CLASS_NAMES[int(c)]: float(v) for c, v in zip(r.box.ap_class_index, r.box.maps[r.box.ap_class_index])},
        "per_class_mAP50": {CLASS_NAMES[int(c)]: float(v) for c, v in zip(r.box.ap_class_index, r.box.ap50)},
        "speed_ms": r.speed,
    }
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=1))
print(json.dumps(metrics, indent=1))

In [ ]:
# Per-image predictions at a very low confidence so the pass/fail threshold can be swept locally
preds = {}
for sp in ("val", "test"):
    paths = df.loc[df.split == sp, "image"].tolist()
    for r in m.predict(paths, imgsz=IMGSZ, conf=0.001, iou=0.6, stream=True, verbose=False):
        b = r.boxes
        preds[pathlib.Path(r.path).name] = [
            {"cls": CLASS_NAMES[int(c)], "xc": float(x[0]), "yc": float(x[1]), "w": float(x[2]),
             "h": float(x[3]), "conf": float(p)}
            for c, x, p in zip(b.cls.tolist(), b.xywhn.tolist(), b.conf.tolist())
        ]
(run_dir / "predictions.json").write_text(json.dumps(preds))
print(len(preds), "images predicted")

In [ ]:
# Export to ONNX (fp32 and fp16) for CPU inference with onnxruntime on a laptop
import shutil
onnx32 = m.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)
shutil.move(onnx32, run_dir / "weights" / "best_fp32.onnx")
# Same weights exported at other input sizes, for the resolution sensitivity study
for sz in (320, 480, 800):
    try:
        f = YOLO(str(best)).export(format="onnx", imgsz=sz, opset=17, simplify=True, dynamic=False)
        shutil.move(f, run_dir / "weights" / f"best_fp32_{sz}.onnx")
    except Exception as exc:
        print("export at", sz, "failed:", exc)
try:
    m16 = YOLO(str(best))
    onnx16 = m16.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False, half=True, device=0)
    shutil.move(onnx16, run_dir / "weights" / "best_fp16.onnx")
except Exception as exc:  # fp16 export is optional (used for the precision comparison)
    print("fp16 export failed:", exc)

# GPU inference timing (median of repeated single-image predictions after warm-up)
import numpy as np
sample = df.loc[df.split == "test", "image"].iloc[0]
for _ in range(10):
    m.predict(sample, imgsz=IMGSZ, verbose=False)
times = []
for _ in range(100):
    t = time.perf_counter(); m.predict(sample, imgsz=IMGSZ, verbose=False); times.append(time.perf_counter() - t)
env["gpu_predict_ms_median"] = 1000 * float(np.median(times))
env["model_info"] = str(m.info(verbose=False))

(run_dir / "env.json").write_text(json.dumps(env, indent=1))
out_zip = shutil.make_archive(f"{DRIVE_DIR}/porksort_runs/{run_name}", "zip", run_dir)
print("Saved", out_zip)

## Done

Download `MyDrive/MMA3001/porksort_runs/yolov8n_<split>.zip`, unzip it into `results/yolo/yolov8n_<split>/`
in the repo, and run `python scripts/04_evaluate_yolo.py`.